# İndikatör Test Notebook

Bu notebook, trading stratejisinde kullanılan teknik indikatörleri test eder.

## İçerik:
1. İndikatör import ve setup
2. ATR (Average True Range) testi
3. SuperTrend testi
4. MOST testi
5. QQE MOD testi
6. RVOL testi
7. Multi-timeframe analiz
8. Sinyal üretimi testi

**Yazar:** Trading Bot Sistemi  
**Faz:** 5  
**Tarih:** 2025-11-12

In [ ]:
# Gerekli kütüphaneler
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Proje root'u path'e ekle
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Trading modülleri
from indicators import (
    calculate_atr,
    calculate_supertrend,
    calculate_most,
    qqe_mod,
    rvol
)

print("✅ Kütüphaneler yüklendi")

## 1. Test Verisi Oluşturma

Gerçekçi market verisi simülasyonu

In [ ]:
# Test parametreleri
n_points = 500
base_price = 50000

# Random walk ile fiyat serisi
np.random.seed(42)
returns = np.random.randn(n_points) * 0.02  # %2 volatilite
close_prices = base_price * np.exp(np.cumsum(returns))

# OHLC verisi oluştur
high_prices = close_prices * (1 + np.random.rand(n_points) * 0.01)
low_prices = close_prices * (1 - np.random.rand(n_points) * 0.01)
volume = np.random.rand(n_points) * 1000000

# DataFrame oluştur
df = pd.DataFrame({
    'open': close_prices,
    'high': high_prices,
    'low': low_prices,
    'close': close_prices,
    'volume': volume
})

print(f"✅ Test verisi oluşturuldu: {len(df)} data points")
print(f"   Fiyat aralığı: ${df['close'].min():.2f} - ${df['close'].max():.2f}")
df.head()

## 2. ATR (Average True Range) Testi

In [ ]:
# ATR hesapla
atr_values = calculate_atr(
    high=df['high'].values,
    low=df['low'].values,
    close=df['close'].values,
    period=14
)

df['atr'] = atr_values

# Görselleştir
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Fiyat grafiği
ax1.plot(df['close'], label='Close Price', linewidth=2)
ax1.set_title('Fiyat Hareketi', fontsize=14)
ax1.set_ylabel('Fiyat (USDT)', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# ATR grafiği
ax2.plot(df['atr'], label='ATR (14)', color='orange', linewidth=2)
ax2.set_title('Average True Range (Volatilite)', fontsize=14)
ax2.set_ylabel('ATR', fontsize=12)
ax2.set_xlabel('Zaman', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 ATR İstatistikleri:")
print(f"   Ortalama: {df['atr'].mean():.2f}")
print(f"   Min: {df['atr'].min():.2f}")
print(f"   Max: {df['atr'].max():.2f}")

## 3. SuperTrend Testi

In [ ]:
# SuperTrend hesapla
st_line, st_trend = calculate_supertrend(
    high=df['high'].values,
    low=df['low'].values,
    close=df['close'].values,
    atr_period=10,
    multiplier=3.0
)

df['supertrend'] = st_line
df['st_trend'] = st_trend

# Görselleştir
plt.figure(figsize=(14, 7))

# Fiyat ve SuperTrend
plt.plot(df['close'], label='Close Price', linewidth=2, color='black')
plt.plot(df['supertrend'], label='SuperTrend', linewidth=2, color='blue', alpha=0.7)

# Trend renklendirmesi
uptrend = df['st_trend'] == 1
downtrend = df['st_trend'] == -1

plt.fill_between(df.index, df['close'].min(), df['close'].max(), 
                 where=uptrend, alpha=0.1, color='green', label='Uptrend')
plt.fill_between(df.index, df['close'].min(), df['close'].max(), 
                 where=downtrend, alpha=0.1, color='red', label='Downtrend')

plt.title('SuperTrend İndikatörü (10, 3.0)', fontsize=14)
plt.ylabel('Fiyat (USDT)', fontsize=12)
plt.xlabel('Zaman', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Trend istatistikleri
uptrend_pct = (df['st_trend'] == 1).sum() / len(df) * 100
downtrend_pct = (df['st_trend'] == -1).sum() / len(df) * 100

print(f"\n📊 SuperTrend İstatistikleri:")
print(f"   Uptrend: {uptrend_pct:.1f}%")
print(f"   Downtrend: {downtrend_pct:.1f}%")

## 4. MOST Testi

In [ ]:
# MOST hesapla
most_line, most_trend = calculate_most(
    close=df['close'].values,
    length=9,
    percent=2.0,
    ma_type='VAR'
)

df['most'] = most_line
df['most_trend'] = most_trend

# Görselleştir
plt.figure(figsize=(14, 7))

plt.plot(df['close'], label='Close Price', linewidth=2, color='black')
plt.plot(df['most'], label='MOST', linewidth=2, color='purple', alpha=0.7)

# Trend renklendirmesi
uptrend = df['most_trend'] == 1
downtrend = df['most_trend'] == -1

plt.fill_between(df.index, df['close'].min(), df['close'].max(), 
                 where=uptrend, alpha=0.1, color='green', label='Uptrend')
plt.fill_between(df.index, df['close'].min(), df['close'].max(), 
                 where=downtrend, alpha=0.1, color='red', label='Downtrend')

plt.title('MOST İndikatörü (9, 2.0%)', fontsize=14)
plt.ylabel('Fiyat (USDT)', fontsize=12)
plt.xlabel('Zaman', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📊 MOST İstatistikleri:")
print(f"   Uptrend: {(df['most_trend'] == 1).sum() / len(df) * 100:.1f}%")
print(f"   Downtrend: {(df['most_trend'] == -1).sum() / len(df) * 100:.1f}%")

## 5. QQE MOD Testi

In [ ]:
# QQE MOD hesapla
qqe_line, qqe_signal = qqe_mod(
    close=df['close'].values,
    rsi_period=6,
    rsi_smoothing=5,
    qqe_factor=3.0,
    threshold=3
)

df['qqe'] = qqe_line
df['qqe_signal'] = qqe_signal

# Görselleştir
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Fiyat grafiği
ax1.plot(df['close'], label='Close Price', linewidth=2, color='black')
ax1.set_title('Fiyat Hareketi', fontsize=14)
ax1.set_ylabel('Fiyat (USDT)', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# QQE MOD grafiği
ax2.plot(df['qqe'], label='QQE Line', linewidth=2, color='blue')
ax2.plot(df['qqe_signal'], label='Signal Line', linewidth=2, color='red', alpha=0.7)
ax2.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
ax2.set_title('QQE MOD İndikatörü', fontsize=14)
ax2.set_ylabel('QQE Değeri', fontsize=12)
ax2.set_xlabel('Zaman', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Crossover sayısı
bullish = (df['qqe'] > df['qqe_signal']).sum()
bearish = (df['qqe'] < df['qqe_signal']).sum()

print(f"\n📊 QQE MOD İstatistikleri:")
print(f"   Bullish: {bullish / len(df) * 100:.1f}%")
print(f"   Bearish: {bearish / len(df) * 100:.1f}%")

## 6. RVOL Testi

In [ ]:
# RVOL hesapla
rvol_values = rvol(
    volume=df['volume'].values,
    close=df['close'].values,
    period=20
)

df['rvol'] = rvol_values

# Görselleştir
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Fiyat grafiği
ax1.plot(df['close'], label='Close Price', linewidth=2, color='black')
ax1.set_title('Fiyat Hareketi', fontsize=14)
ax1.set_ylabel('Fiyat (USDT)', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# RVOL grafiği
colors = ['green' if v > 1.5 else 'red' if v < 0.5 else 'gray' for v in df['rvol']]
ax2.bar(df.index, df['rvol'], color=colors, alpha=0.6)
ax2.axhline(y=1.0, color='blue', linestyle='--', label='Normal (1.0x)')
ax2.axhline(y=1.5, color='green', linestyle='--', alpha=0.5, label='Yüksek (1.5x)')
ax2.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Düşük (0.5x)')
ax2.set_title('Relative Volume (RVOL)', fontsize=14)
ax2.set_ylabel('RVOL', fontsize=12)
ax2.set_xlabel('Zaman', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# İstatistikler
high_vol = (df['rvol'] > 1.5).sum() / len(df) * 100
low_vol = (df['rvol'] < 0.5).sum() / len(df) * 100
normal_vol = 100 - high_vol - low_vol

print(f"\n📊 RVOL İstatistikleri:")
print(f"   Yüksek Hacim (>1.5x): {high_vol:.1f}%")
print(f"   Normal Hacim: {normal_vol:.1f}%")
print(f"   Düşük Hacim (<0.5x): {low_vol:.1f}%")

## 7. SignalGenerator Testi

In [ ]:
from trading import SignalGenerator

# SignalGenerator oluştur
signal_gen = SignalGenerator()

# Test verisi hazırla (son 100 nokta)
test_idx = -100

data_1h = {
    'high': df['high'].values[test_idx:],
    'low': df['low'].values[test_idx:],
    'close': df['close'].values[test_idx:]
}

data_15m = {
    'high': df['high'].values[test_idx:],
    'low': df['low'].values[test_idx:],
    'close': df['close'].values[test_idx:],
    'volume': df['volume'].values[test_idx:]
}

# Sinyal üret
signal = signal_gen.generate_signal(
    data_1h=data_1h,
    data_15m=data_15m,
    timestamp='2025-11-12T00:00:00'
)

print("\n" + "=" * 60)
print("📊 SINYAL SONUCU")
print("=" * 60)
print(f"Sinyal Tipi: {signal.signal_type.value}")
print(f"Güven Seviyesi: {signal.confidence.value}")
print(f"Güven Skoru: {signal.confidence_score:.2%}")
print(f"Entry Fiyatı: ${signal.entry_price:.2f}")
print(f"Stop Loss: ${signal.stop_loss:.2f}" if signal.stop_loss else "Stop Loss: None")
print(f"Take Profit: ${signal.take_profit:.2f}" if signal.take_profit else "Take Profit: None")

print("\n📈 İndikatör Sinyalleri:")
for ind in signal.indicators:
    print(f"  {ind.name:15s} ({ind.timeframe}): {ind.signal:8s} - {ind.reason}")

print("\n💡 Nedenler:")
for reason in signal.reasons:
    print(f"  • {reason}")
print("=" * 60)

## Sonuç

Bu notebook'ta:
- ✅ Tüm indikatörler test edildi
- ✅ Görselleştirmeler oluşturuldu
- ✅ SignalGenerator ile sinyal üretimi test edildi

Sonraki adımlar:
- Gerçek market verisi ile test
- Farklı parametre optimizasyonu
- Backtest analizi